# Minimal GCN disease-gene prioritization

This notebook trains a two-layer graph convolutional network on the **loaded PPI graph** for one disease. It uses the known disease genes as positives, holds some out for validation and testing, and samples unknown genes as training negatives.

The input features are deliberately minimal: a constant, standardized log-degree, and an indicator for the training disease genes. The GCN propagates these features over the PPI. Unknown genes are not confirmed negatives, so the resulting metrics demonstrate the workflow rather than establish biological performance.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from bioGraph.gcn_prioritization.main import (
    load_disease_genes,
    load_ppi_graph,
    train_single_disease,
)
from bioGraph.performance_metric import average_precision_at_k, recall_at_k

## 1. Load the biological graph and one disease

Node labels are Entrez gene IDs. Edges are undirected protein-protein interactions; the symbol is retained as a node attribute for readable output.

In [2]:
data_dir = Path.cwd()
graph = load_ppi_graph(data_dir / "PPI202207.txt")
diseases = load_disease_genes(data_dir / "pcbi.1004120.s004.txt")

disease_name = "breast neoplasms"
disease_genes = diseases[disease_name]

print(f"Graph: {graph.number_of_nodes():,} genes, {graph.number_of_edges():,} interactions")
print(f"{disease_name}: {len(disease_genes)} known genes before graph filtering")

Graph: 17,504 genes, 354,647 interactions
breast neoplasms: 40 known genes before graph filtering


## 2. Train

Only `train_genes` enter the positive training labels and seed-indicator feature. Validation genes select the early-stopping checkpoint. Test genes remain hidden until the final ranking is evaluated. Sparse graph multiplication keeps the full PPI practical in memory.

In [3]:
result = train_single_disease(
    graph,
    disease_genes,
    hidden_dim=32,
    epochs=100,
    learning_rate=0.01,
    negative_ratio=5,
    patience=20,
    seed=42,
)

print(
    f"Known genes in/outside graph: {result['known_in_graph']}/{result['known_not_in_graph']}\n"
    f"Split: {len(result['train_genes'])} train, {len(result['val_genes'])} validation, "
    f"{len(result['test_genes'])} test\n"
    f"Ran {len(result['losses'])} epochs on {result['device']}; "
    f"best validation AP = {result['best_val_ap']:.4f}"
)

Known genes in/outside graph: 40/0
Split: 26 train, 6 validation, 8 test
Ran 29 epochs on cpu; best validation AP = 0.0010


## 3. Evaluate and inspect candidates

Training and validation positives are removed from the final candidate list. Average precision rewards placing hidden test genes early throughout the ranking; recall@k reports the fraction recovered in the first k candidates.

In [4]:
ranking = result["ranking"]
test_genes = result["test_genes"]

print(f"Test average precision: {average_precision_at_k(ranking, test_genes):.4f}")
for k in (25, 100, 300):
    print(f"Recall@{k}: {recall_at_k(ranking, test_genes, k):.4f}")

ranking[:10]

Test average precision: 0.0006
Recall@25: 0.0000
Recall@100: 0.0000
Recall@300: 0.0000


[{'gene_id': 6137, 'symbol': 'RPL13', 'score': 0.5906106233596802},
 {'gene_id': 6122, 'symbol': 'RPL3', 'score': 0.5880118608474731},
 {'gene_id': 11224, 'symbol': 'RPL35', 'score': 0.5878942608833313},
 {'gene_id': 6124, 'symbol': 'RPL4', 'score': 0.5866166949272156},
 {'gene_id': 6431, 'symbol': 'SRSF6', 'score': 0.5851396918296814},
 {'gene_id': 4113, 'symbol': 'MAGEB2', 'score': 0.5845100283622742},
 {'gene_id': 11318, 'symbol': 'GPR182', 'score': 0.5844327807426453},
 {'gene_id': 6143, 'symbol': 'RPL19', 'score': 0.5840274691581726},
 {'gene_id': 3192, 'symbol': 'HNRNPU', 'score': 0.5837318301200867},
 {'gene_id': 6194, 'symbol': 'RPS6', 'score': 0.583268940448761}]